# LLM Building Blocks

Master embeddings, positional encoding, and the self-attention mechanism that forms the heart of transformer models. Understand token representation, why attention is permutation-invariant, and the mathematical foundations of modern LLMs.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/llm-building-blocks)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Colab provides PyTorch with CUDA support, so these notebooks run on a real GPU rather than Apple's MPS backend. If a code sample uses `device = torch.device('mps' ...)`, it will fall back to `cpu` on Colab unless you adapt it; replace `'mps'` with `'cuda'` for GPU execution.


In [ ]:
!pip install torch numpy

---

## Lesson examples (optional)

These are the runnable code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.


### Lesson example: Embeddings & Positional Encoding


In [ ]:

import torch
import torch.nn as nn

# ===== Embedding Layer Example =====
vocab_size = 10000
embedding_dim = 768
batch_size = 2
seq_len = 10

# Token embedding layer (PyTorch built-in)
token_embedding = nn.Embedding(vocab_size, embedding_dim)

# Create sample token IDs
token_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
print(f"Token IDs shape: {token_ids.shape}")  # (2, 10)

# Get embeddings
token_vectors = token_embedding(token_ids)
print(f"Token vectors shape: {token_vectors.shape}")  # (2, 10, 768)

# ===== Positional Embeddings =====
max_seq_length = 2048
position_embedding = nn.Embedding(max_seq_length, embedding_dim)

# Create position indices
positions = torch.arange(seq_len, dtype=torch.long)
print(f"Positions: {positions}")

# Get position embeddings
pos_vectors = position_embedding(positions)
print(f"Position vectors shape: {pos_vectors.shape}")  # (10, 768)

# ===== Combined Embeddings =====
# Broadcasting: token (2, 10, 768) + position (10, 768) → (2, 10, 768)
combined = token_vectors + pos_vectors
print(f"Combined embeddings shape: {combined.shape}")  # (2, 10, 768)

# ===== Verify Gradient Flow =====
combined.sum().backward()
print(f"Token embedding grad exists: {token_embedding.weight.grad is not None}")
print(f"Position embedding grad exists: {position_embedding.weight.grad is not None}")

# ===== Demonstrating Permutation Invariance Problem =====

class DotProductAttention(nn.Module):
    """Simplified attention to show permutation issue"""
    def __init__(self, embed_dim):
        super().__init__()
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        Q = self.W_q(x)  # (batch, seq, dim)
        K = self.W_k(x)  # (batch, seq, dim)
        scores = torch.matmul(Q, K.transpose(-2, -1))  # (batch, seq, seq)
        return scores

# Create attention module
attn = DotProductAttention(embedding_dim)

# Original sequence attention
scores_original = attn(combined)
print(f"\nOriginal attention scores shape: {scores_original.shape}")

# Permuted sequence: shuffle positions 1 and 3
perm_indices = torch.tensor([0, 3, 2, 1, 4, 5, 6, 7, 8, 9])
combined_permuted = combined[0, perm_indices, :].unsqueeze(0)
scores_permuted = attn(combined_permuted)

print(f"Permuted attention scores shape: {scores_permuted.shape}")

# Without positional encoding, scores are permuted accordingly
# But WITH positional encoding, order is respected!

# ===== Initialize Embeddings Properly =====
def init_embedding(embed_layer, dim):
    """Initialize embedding with proper variance"""
    with torch.no_grad():
        embed_layer.weight.normal_(mean=0, std=1.0 / dim ** 0.5)
    return embed_layer

token_embedding_initialized = init_embedding(token_embedding, embedding_dim)
print(f"\n✓ Embeddings initialized with variance ~ 1/√d = {1.0 / embedding_dim ** 0.5:.4f}")
        

### Lesson example: Self-Attention Mechanism


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ===== Self-Attention with Dimension Tracking =====

class SelfAttention(nn.Module):
    def __init__(self, embed_dim, head_dim=64):
        super().__init__()
        self.embed_dim = embed_dim
        self.head_dim = head_dim

        # Q, K, V projections
        self.W_q = nn.Linear(embed_dim, head_dim)
        self.W_k = nn.Linear(embed_dim, head_dim)
        self.W_v = nn.Linear(embed_dim, head_dim)

        # Scaling factor
        self.scale = 1.0 / math.sqrt(head_dim)

    def forward(self, x, mask=None):
        """
        Args:
            x: (batch, seq_len, embed_dim)
            mask: (seq_len, seq_len) with 1=attend, 0=mask

        Returns:
            output: (batch, seq_len, head_dim)
            weights: (batch, seq_len, seq_len)
        """
        batch_size, seq_len, _ = x.shape

        # Project to Q, K, V
        Q = self.W_q(x)  # (batch, seq_len, head_dim)
        K = self.W_k(x)  # (batch, seq_len, head_dim)
        V = self.W_v(x)  # (batch, seq_len, head_dim)

        # Compute attention scores: Q @ K^T / √head_dim
        # (batch, seq_len, head_dim) @ (batch, head_dim, seq_len)
        # = (batch, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        print(f"Scores shape: {scores.shape}")
        print(f"Scores variance: {scores.var().item():.4f}")  # Should be ~1

        # Apply causal mask if provided
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # Apply softmax to get attention weights
        weights = F.softmax(scores, dim=-1)

        # Handle NaN from softmax of -inf
        weights = torch.nan_to_num(weights, 0.0)
        print(f"Weights shape: {weights.shape}")
        print(f"Weights sum per query: {weights.sum(dim=-1)[0, 0].item():.4f}")  # Should be ~1

        # Aggregate values: weights @ V
        # (batch, seq_len, seq_len) @ (batch, seq_len, head_dim)
        # = (batch, seq_len, head_dim)
        output = torch.matmul(weights, V)

        return output, weights

# ===== Create Causal Mask =====

def create_causal_mask(seq_len):
    """Lower triangular mask: position i can attend to j where j <= i"""
    # torch.tril returns lower triangular: [[1,0,0], [1,1,0], [1,1,1]]
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask

# ===== Test =====

embed_dim = 512
head_dim = 64
seq_len = 8
batch_size = 2

attention = SelfAttention(embed_dim, head_dim)
x = torch.randn(batch_size, seq_len, embed_dim)

# Without mask (bidirectional)
print("\n=== Bidirectional Attention ===")
output_bi, weights_bi = attention(x)
print(f"Output shape: {output_bi.shape}")

# With causal mask
print("\n=== Causal Attention ===")
mask = create_causal_mask(seq_len)
output_causal, weights_causal = attention(x, mask=mask)

# Verify causal structure
print(f"\nPosition 0 attention weights (should only attend to itself):")
print(weights_causal[0, 0, :])  # Should be [1, 0, 0, ...]

print(f"\nPosition 5 attention weights (can attend to 0-5):")
print(weights_causal[0, 5, :])  # Should be [≈0.2, ≈0.2, ..., ≈0.2, 0, 0, 0]

# ===== Demonstrate Scaling Importance =====

print("\n=== Scaling Impact on Gradients ===")

# Without scaling
Q = torch.randn(100, head_dim)
K = torch.randn(100, head_dim)
scores_unscaled = torch.matmul(Q, K.t())
attn_unscaled = F.softmax(scores_unscaled, dim=-1)

# With scaling
scores_scaled = scores_unscaled / math.sqrt(head_dim)
attn_scaled = F.softmax(scores_scaled, dim=-1)

print(f"Unscaled scores variance: {scores_unscaled.var().item():.2f}")  # ~64
print(f"Scaled scores variance: {scores_scaled.var().item():.2f}")      # ~1
print(f"\nUnscaled attention max: {attn_unscaled.max().item():.4f}")   # Nearly one-hot
print(f"Scaled attention max: {attn_scaled.max().item():.4f}")         # More distributed

# Gradient norms
attn_unscaled.sum().backward()
grad_unscaled = Q.grad.norm()

Q.grad = None
attn_scaled.sum().backward()
grad_scaled = Q.grad.norm()

print(f"\nGradient norm (unscaled): {grad_unscaled.item():.6f}")
print(f"Gradient norm (scaled): {grad_scaled.item():.6f}")
print("✓ Scaled version has better gradient flow")
        

---

## Exercise: Implement Self-Attention from Scratch


Build a complete self-attention module including: (1) Q, K, V projections, (2) scaled dot-product attention computation, (3) causal masking, (4) softmax with proper handling of masked positions, and (5) output aggregation. Your implementation must handle batches correctly, produce correct gradient flow, and match reference behavior. Test against a provided reference implementation.


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SelfAttentionFromScratch(nn.Module):
    """
    Implement self-attention completely from scratch.

    Args:
        embedding_dim: Input embedding dimension
        head_dim: Output dimension per head
    """
    def __init__(self, embedding_dim, head_dim=64):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.head_dim = head_dim

        # TODO: Create Q, K, V projection layers as nn.Linear
        # Each should map from embedding_dim → head_dim

        # TODO: Compute and store the scaling factor 1/√head_dim

    def forward(self, x, mask=None):
        """
        Args:
            x: (batch_size, seq_len, embedding_dim)
            mask: (seq_len, seq_len) where 1=attend, 0=mask; or None

        Returns:
            output: (batch_size, seq_len, head_dim)
            attention_weights: (batch_size, seq_len, seq_len) for visualization
        """
        batch_size, seq_len, _ = x.shape

        # TODO: Project x to Q, K, V using the learned projections
        # Q, K, V should all be shape (batch_size, seq_len, head_dim)

        # TODO: Compute attention scores as: Q @ K^T / √head_dim
        # Result should be (batch_size, seq_len, seq_len)
        # Hint: Use torch.matmul and transpose(-2, -1)

        # TODO: Apply causal mask if provided
        # Hint: scores.masked_fill(mask == 0, float('-inf'))

        # TODO: Apply softmax over the last dimension (key dimension)
        # Handle NaN from softmax(-inf) using torch.nan_to_num(weights, 0.0)

        # TODO: Compute output as: attention_weights @ V
        # Result should be (batch_size, seq_len, head_dim)

        return output, attention_weights

# ===== Test Code =====
if __name__ == "__main__":
    embedding_dim = 512
    head_dim = 64
    seq_len = 8
    batch_size = 2

    # Create attention layer
    attention = SelfAttentionFromScratch(embedding_dim, head_dim)

    # Create input and causal mask
    x = torch.randn(batch_size, seq_len, embedding_dim)
    mask = torch.tril(torch.ones(seq_len, seq_len))

    # Forward pass
    output, weights = attention(x, mask=mask)

    # Verify output shapes
    assert output.shape == (batch_size, seq_len, head_dim), f"Output shape {output.shape} != {(batch_size, seq_len, head_dim)}"
    assert weights.shape == (batch_size, seq_len, seq_len), f"Weights shape {weights.shape} != {(batch_size, seq_len, seq_len)}"

    # Verify causal structure: position 0 only attends to itself
    assert weights[0, 0, 1:].max().item() < 1e-5, "Position 0 should only attend to itself"
    assert weights[0, -1, :].sum().item() > 0.99, "All attention weights should sum to 1"

    # Verify gradient flow
    loss = output.sum()
    loss.backward()
    assert attention.W_q.weight.grad is not None, "Gradients must flow through Q projection"

    print("✓ All assertions passed!")
      

---

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/llm-building-blocks) for the solution and discussion.
